# Migrating from Document AI to AI_EXTRACT

This demo shows how to migrate from the deprecated **Document AI** (`model!PREDICT`) to the new **AI_EXTRACT** function.

In [ ]:
SET DB_NAME = 'DOC_AI_DEPRECATION_DEMO';
SET SCHEMA_NAME = 'PUBLIC';

In [ ]:
CREATE DATABASE IF NOT EXISTS IDENTIFIER($DB_NAME);
USE DATABASE IDENTIFIER($DB_NAME);

CREATE SCHEMA IF NOT EXISTS IDENTIFIER($SCHEMA_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

In [ ]:
CREATE STAGE IF NOT EXISTS DEPRECATED_DOC_AI_IMAGES
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

CREATE STAGE IF NOT EXISTS DEMO_DOCS
    DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
    ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

MY_STAGE = 'DEMO_DOCS'
MY_FILE_NAME = 'data/*'

put_result = session.file.put(MY_FILE_NAME, MY_STAGE, auto_compress=False, overwrite=True)

In [ ]:
ALTER STAGE DEMO_DOCS REFRESH;
SELECT * FROM DIRECTORY(@DEMO_DOCS);

---
## The Old Way: Document AI with model!PREDICT

**Problems with the old approach:**
- Required creating and training a model in Snowsight UI
- Fine-tuning needed for good accuracy
- Limited flexibility in question formats

In [ ]:
SELECT 
    DEPRECATION_DEMO!PREDICT(
        GET_PRESIGNED_URL(@DEMO_DOCS, relative_path),
        1
    ) AS predictions
FROM DIRECTORY(@DEMO_DOCS)
WHERE relative_path = 'Manual_2022-02-01.pdf'
LIMIT 1;

---
## Migration Path: Preserving Your Fine-Tuned Model

If you have a fine-tuned Document AI model you want to preserve, follow these steps:

1. Go to AI Studio -> Document Processing Playground -> Document AI Builds -> Migrate Now
2. Convert Current Pipelines to use AI_EXTRACT() instead of MODEL!PREDICT()
3. (Optional)Select a Model and Export to a Stage (If you want to access priorly trained on documents with their prompts and responses)

In [ ]:
SELECT AI_EXTRACT(
    MODEL => 'DEPRECATION_DEMO',
    FILE => TO_FILE('@DEMO_DOCS', 'Manual_2022-02-01.pdf')
) AS result;

In [ ]:
CREATE OR REPLACE FILE FORMAT my_json
  TYPE = 'JSON';

In [ ]:
CREATE OR REPLACE TABLE exported_data_table AS (
   SELECT
      input_file.$1:file AS file,
      input_file.$1:prompt AS prompt,
      input_file.$1:annotatedResponse AS response
   FROM '@DEPRECATED_DOC_AI_IMAGES/DOC_AI_DEPRECATION_DEMO_PUBLIC_DEPRECATION_DEMO_2026_01_28_15_48_27/annotations.jsonl' (FILE_FORMAT => my_json) input_file
   WHERE response != '{}'
);

In [ ]:
SELECT
    *
FROM EXPORTED_DATA_TABLE;

---
## The New Way: AI_EXTRACT

AI_EXTRACT is a **zero-shot** function — no model training required. Just ask questions!

## Document 1: Simple Gene Table (CpG Sites)

A straightforward 4-column table with gene names, numeric values, and text descriptions.

In [ ]:
from PIL import Image
import os

img_path = os.path.join(os.getcwd(), "data", "simple_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'simple_table_data.jpg'),
    responseFormat => {
        'schema': {
            'type': 'object',
            'properties': {
                'gene_insight_table': {
                    'description': 'CpG Sites and genes of interest(Genes with significant TSS200',
                    'type': 'object',
                    'column_ordering': ['gene', 'case', 'control', 'ipa_network'],
                    'properties': {
                        'gene': {'type': 'array'},
                        'case': {'type': 'array'},
                        'control': {'type': 'array'},
                        'ipa_network': {'type': 'array'}
                    }
                }
            }
        }
    }
) AS extracted_genes;

In [ ]:
import json
result = json.loads(cells.table_inference.to_pandas()['EXTRACTED_GENES'][0])
print(json.dumps(result, indent=2))

## Document 2: Dual-Column Scientific Table (Gene Sequence Identity)

A table with gene identifiers and multiple numeric columns showing identity percentages and KaKs values.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "dual_column_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
CREATE OR REPLACE TABLE prompt_templates (
    template_id VARCHAR PRIMARY KEY,
    response_format VARIANT
);

In [ ]:
INSERT INTO PROMPT_TEMPLATES
  SELECT 
      'DUAL_SEQUENCE_TABLE',
      PARSE_JSON($$
      {
        'schema': {
            'type': 'object',
            'properties': {
                'sequence_id_table': {
                    'description': 'Interspecific sequence identities and Ka/Ks values for seven genes',
                    'type': 'object',
                    'column_ordering': ['gene_identifier', 'identity_rice_maize', 'identity_rice_sorghum', 'identity_sorghum_maize', 'kaks_rice_maize', 'kaks_rice_sorghum', 'kaks_sorghum_maize', 'size'],
                    'properties': {
                        'gene_identifier': {'type': 'array'},
                        'identity_rice_maize': {'type': 'array'},
                        'identity_rice_sorghum': {'type': 'array'},
                        'identity_sorghum_maize': {'type': 'array'},
                        'kaks_rice_maize': {'type': 'array'},
                        'kaks_rice_sorghum': {'type': 'array'},
                        'kaks_sorghum_maize': {'type': 'array'},
                        'size': {'type': 'array'}
                    }
                }
            }
        }
    }
      $$);

In [ ]:
CREATE TEMPORARY TABLE DUAL_COLUMN_TABLE as
SELECT AI_EXTRACT(
    file => TO_FILE('@DEMO_DOCS', 'dual_column_table_data.jpg'),
    responseFormat => (
        SELECT response_format 
        FROM prompt_templates
        WHERE template_id = 'DUAL_SEQUENCE_TABLE'
    )
) AS extracted_sequence_ids;

In [ ]:
SELECT 
      f.index AS row_num,
      f.value::STRING AS gene_identifier,
      t.extracted_sequence_ids:response:sequence_id_table:identity_rice_maize[f.index]::STRING AS identity_rice_maize,
      t.extracted_sequence_ids:response:sequence_id_table:identity_rice_sorghum[f.index]::STRING AS identity_rice_sorghum,
      t.extracted_sequence_ids:response:sequence_id_table:identity_sorghum_maize[f.index]::STRING AS identity_sorghum_maize,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_rice_maize[f.index]::STRING AS kaks_rice_maize,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_rice_sorghum[f.index]::STRING AS kaks_rice_sorghum,
      t.extracted_sequence_ids:response:sequence_id_table:kaks_sorghum_maize[f.index]::STRING AS kaks_sorghum_maize,
      t.extracted_sequence_ids:response:sequence_id_table:size[f.index]::STRING AS size
  FROM DUAL_COLUMN_TABLE t,
  LATERAL FLATTEN(input => t.extracted_sequence_ids:response:sequence_id_table:gene_identifier) f

## Document 3: Multi-Layer Nested Table (Medical Demographics)

A complex table with hierarchical categories (Age group, FIGO, Morphology, Surgery, Radiotherapy) and multiple population columns.

In [ ]:
img_path = os.path.join(os.getcwd(), "data", "multi_layer_nested_table_data.jpg")
img = Image.open(img_path)
img

In [ ]:
CREATE TEMPORARY TABLE MULTI_LAYER_NESTED_TABLE_DATA_FLATTENED as
SELECT AI_EXTRACT(
      TO_FILE('@DEMO_DOCS', 'multi_layer_nested_table_data.jpg'),
      {
        'schema': {
            'type': 'object',
            'properties': {
                'tumor_characteristics': {
                  'type': 'object',
                  'column_ordering': ['category', 'subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct', 'p_value'],
                  'properties': {
                      'category': {'type': 'array', 'description': 'Main category: Age group, FIGO, Morphology, Surgery, or Radiotherapy'},
                      'subcategory': {'type': 'array', 'description': 'Subcategory value within the main category'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'},
                      'p_value': {'type': 'array', 'description': 'p-value applies to entire category group'}
                  }
              }        
           }
        }
      }
  ) AS extracted_tumor_data;

In [ ]:
SELECT 
      f.index AS row_num,
      f.value::STRING AS category,
      t.extracted_tumor_data:response:tumor_characteristics:subcategory[f.index]::STRING AS subcategory,
      t.extracted_tumor_data:response:tumor_characteristics:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:tumor_characteristics:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:tumor_characteristics:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:tumor_characteristics:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:tumor_characteristics:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:tumor_characteristics:caucasian_pct[f.index]::STRING AS caucasian_pct,
      t.extracted_tumor_data:response:tumor_characteristics:p_value[f.index]::STRING AS p_value
  FROM MULTI_LAYER_NESTED_TABLE_DATA_FLATTENED t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:tumor_characteristics:category) f

In [ ]:
CREATE TEMPORARY TABLE MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE as
SELECT AI_EXTRACT(
      TO_FILE('@DEMO_DOCS', 'multi_layer_nested_table_data.jpg'),
      {
        'schema': {
            'type': 'object',
            'properties': {
                'age_group': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Age ranges: < 40, 40-49, 50-59, 60-69, 70+'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'figo': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'FIGO stages: I, II, III, IV, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'morphology': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Morphology types: Serous, Clear cell, Endometrioid, Mucinous, Others, NOS'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'surgery': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Surgery status: With surgery, Without surgery, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'radiotherapy': {
                  'type': 'object',
                  'column_ordering': ['subcategory', 'philippine_freq', 'philippine_pct', 'filipino_american_freq', 'filipino_american_pct', 'caucasian_freq', 'caucasian_pct'],
                  'properties': {
                      'subcategory': {'type': 'array', 'description': 'Radiotherapy status: With radiotherapy, Without radiotherapy, Unknown'},
                      'philippine_freq': {'type': 'array'},
                      'philippine_pct': {'type': 'array'},
                      'filipino_american_freq': {'type': 'array'},
                      'filipino_american_pct': {'type': 'array'},
                      'caucasian_freq': {'type': 'array'},
                      'caucasian_pct': {'type': 'array'}
                  }
              },
              'p_values': {
                  'type': 'object',
                  'properties': {
                      'age_group': {'type': 'array'},
                      'figo': {'type': 'array'},
                      'morphology': {'type': 'array'},
                      'surgery': {'type': 'array'},
                      'radiotherapy': {'type': 'array'}
                  }
              }
            }
        }
      }
  ) AS extracted_tumor_data;

In [ ]:
SELECT 
      'Age group' AS category,
      f.value::STRING AS subcategory,
      t.extracted_tumor_data:response:age_group:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:age_group:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:age_group:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:age_group:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:age_group:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:age_group:caucasian_pct[f.index]::STRING AS caucasian_pct
  FROM MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:age_group:subcategory) f

In [ ]:
SELECT 
      'FIGO' AS category,
      f.value::STRING AS subcategory,
      t.extracted_tumor_data:response:figo:philippine_freq[f.index]::INTEGER AS philippine_freq,
      t.extracted_tumor_data:response:figo:philippine_pct[f.index]::STRING AS philippine_pct,
      t.extracted_tumor_data:response:figo:filipino_american_freq[f.index]::INTEGER AS filipino_american_freq,
      t.extracted_tumor_data:response:figo:filipino_american_pct[f.index]::STRING AS filipino_american_pct,
      t.extracted_tumor_data:response:figo:caucasian_freq[f.index]::INTEGER AS caucasian_freq,
      t.extracted_tumor_data:response:figo:caucasian_pct[f.index]::STRING AS caucasian_pct
  FROM MULTI_LAYER_NESTED_TABLE_DATA_SEPARATE t,
  LATERAL FLATTEN(input => t.extracted_tumor_data:response:figo:subcategory) f

## Document 4: Purchase Order

Mix table data with other individual data points

In [ ]:
from pdf2image import convert_from_path

images = convert_from_path("data/purchase_order.pdf")
img = images[0]  # First page as PIL Image
img

In [ ]:
CREATE OR REPLACE TEMPORARY TABLE PURCHASE_ORDER as
SELECT AI_EXTRACT(
      TO_FILE('@DEMO_DOCS', 'purchase_order.pdf'),
      {
        'schema': {
            'type': 'object',
            'properties': {
                'vendor_name': {'type': 'string'},
                'sales_person': {'type': 'string'},
                'vendor_address': {'type': 'string'},
                'vendor_contact_number': {'type': 'string'},
                'vendor_email_address': {'description': 'If <email_address> or empty, return an empty string', 'type': 'string'},
                'customer_name': {'type': 'string'},
                'contact_person': {'type': 'string'},
                'customer_address': {'type': 'string'},
                'customer_contact_number': {'type': 'string'},
                'customer_email_address': {'type': 'string'},
                'tax_percent': {'type': 'string'},
                'tax_total': {'type': 'string'},
                'discount_percent': {'type': 'string'},
                'discount_total': {'type': 'string'},
                'total': {'type': 'string'},
                'includes_authorized_signature': {'type': 'string'},
                'additional_notes': {'type': 'string'},
                'order_line_items': {
                      'type': 'object',
                      'column_ordering': ['item_number', 'details', 'unit', 'quantity', 'unit_price', 'total'],
                      'properties': {
                          'item_number': {'type': 'array'},
                          'details': {'type': 'array'},
                          'unit': {'type': 'array'},
                          'quantity': {'type': 'array'},
                          'unit_price': {'type': 'array'},
                          'total': {'type': 'array'}
                      }
                  }                
           }
        }
      }
  ) AS extracted_po;

In [ ]:
SELECT
    extracted_po:response:vendor_name::STRING AS vendor_name,
    extracted_po:response:sales_person::STRING AS sales_person,
    extracted_po:response:vendor_address::STRING AS vendor_address,
    extracted_po:response:vendor_contact_number::STRING AS vendor_contact_number,
    extracted_po:response:vendor_email_address::STRING AS vendor_email_address,
    extracted_po:response:customer_name::STRING AS customer_name,
    extracted_po:response:contact_person::STRING AS contact_person,
    extracted_po:response:customer_address::STRING AS customer_address,
    extracted_po:response:customer_contact_number::STRING AS customer_contact_number,
    extracted_po:response:customer_email_address::STRING AS customer_email_address,
    extracted_po:response:tax_percent::NUMBER AS tax_percent,
    extracted_po:response:tax_total::NUMBER(38,2) AS tax_total,
    extracted_po:response:discount_percent::NUMBER AS discount_percent,
    extracted_po:response:discount_total::NUMBER(38,2) AS discount_total,
    extracted_po:response:total::NUMBER AS total,
    extracted_po:response:includes_authorized_signature AS includes_authorized_signature,
    extracted_po:response:additional_notes::STRING AS additional_notes
FROM PURCHASE_ORDER;

In [ ]:
SELECT
        po.extracted_po:response:vendor_name::STRING AS vendor_name,
        po.extracted_po:response:customer_name::STRING AS customer_name,
        li.INDEX AS line_item_index,
        po.extracted_po:response:order_line_items:item_number[li.INDEX]::STRING AS item_number,
        po.extracted_po:response:order_line_items:details[li.INDEX]::STRING AS details,
        po.extracted_po:response:order_line_items:unit[li.INDEX]::STRING AS unit,
        po.extracted_po:response:order_line_items:quantity[li.INDEX]::STRING AS quantity,
        po.extracted_po:response:order_line_items:unit_price[li.INDEX]::STRING AS unit_price,
        po.extracted_po:response:order_line_items:total[li.INDEX]::STRING AS total
    FROM PURCHASE_ORDER po,
    LATERAL FLATTEN(input => po.extracted_po:response:order_line_items:item_number) li;

## Cortex Fine-Tuning API for Arctic-Extract (PuPr)

If necessary, move from the zero-shot approach to fine-tuning to create a fine-tuned model based off of Arctic-Extract.

In [ ]:
CREATE OR REPLACE TABLE equipment_inspection_training (f FILE, p VARCHAR, r VARCHAR);

In [ ]:
INSERT INTO equipment_inspection_training
SELECT
      TO_FILE('@DEMO_DOCS', input_file.$1:file),
      concat('{ "schema": {"type": "object", "properties": ', parse_json(input_file.$1:prompt):properties, '}}'),
      input_file.$1:annotatedResponse as response
   FROM '@DEPRECATED_DOC_AI_IMAGES/DOC_AI_DEPRECATION_DEMO_PUBLIC_DEPRECATION_DEMO_2026_01_28_15_48_27/annotations.jsonl' (FILE_FORMAT => my_json) input_file
   WHERE response != '{}'

In [ ]:
select * from equipment_inspection_training

In [ ]:
CREATE OR REPLACE DATASET equipment_inspection_ds;

In [ ]:
ALTER DATASET equipment_inspection_ds
ADD VERSION 'v1' FROM (
  SELECT FL_GET_STAGE(f) || '/' || FL_GET_RELATIVE_PATH(f) AS "file",
       p AS "prompt",
       r AS "response"
  FROM equipment_inspection_training
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.FINETUNE(
  'CREATE',
  'manual_inspection_cortex_tuned',
  'arctic-extract',
  'snow://dataset/DOC_AI_DEPRECATION_DEMO.PUBLIC.equipment_inspection_ds/versions/v1' --TESTING
  -- 'snow://dataset/DOC_AI_DEPRECATION_DEMO.PUBLIC.equipment_inspection_ds/versions/v2' --VALIDATION
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.FINETUNE(
  'DESCRIBE',
  'ft_e19b2290-df6b-481b-9da6-c2e4697107d8'
)

In [ ]:
SELECT AI_EXTRACT(
  model => 'MANUAL_INSPECTION_CORTEX_TUNED',
  file => TO_FILE('@DEMO_DOCS', 'Manual_2022-02-01.pdf')
);

## Full Document Processing Pipeline

Step by step walkthrough of putting this pipeline from start to finalized data, based off of another Snowflake walk through found at this [link](https://www.snowflake.com/en/developers/guides/create-a-document-processing-pipeline-with-ai-extract/)

In [ ]:
-- STEP 1: Create and Fill Prompt Management Table

CREATE TABLE IF NOT EXISTS prompt_templates (
    template_id VARCHAR PRIMARY KEY,
    response_format VARIANT
);

 INSERT INTO prompt_templates
  SELECT 
      'INSPECTION_REVIEWS',
      PARSE_JSON($$
      {
          "schema": {
              "type": "object",
              "properties": {
                  "list_of_units": {
                      "description": "Extract the table showing all units and their reported conditions",
                      "type": "object",
                      "column_ordering": ["unit_name", "condition"],
                      "properties": {
                          "unit_name": {
                              "description": "Name of the unit",
                              "type": "array"
                          },
                          "condition": {
                              "description": "Condition reported for the unit",
                              "type": "array"
                          }
                      }
                  },
                  "inspection_date": {
                      "description": "What is the inspection date?",
                      "type": "string"
                  },
                  "inspection_grade": {
                      "description": "What is the grade?",
                      "type": "string"
                  },
                  "inspector": {
                      "description": "Who performed the inspection?",
                      "type": "string"
                  }
              }
          }
      }
      $$);

-- STEP 2: AI_EXTRACT Wrapper Function

CREATE OR REPLACE FUNCTION extract_document_data(
    stage_name STRING,
    file_path STRING,
    template_id STRING
)
RETURNS VARIANT
LANGUAGE SQL
AS
$$
    SELECT AI_EXTRACT(
        file => TO_FILE(stage_name, file_path),
        responseFormat => (
            SELECT response_format 
            FROM prompt_templates
            WHERE template_id = template_id
        )
    ):response
$$;

-- STEP 3: Create Table to Store Reviews

CREATE TABLE IF NOT EXISTS pdf_reviews (
    file_name VARCHAR,
    file_size VARIANT,
    last_modified VARCHAR,
    snowflake_file_url VARCHAR,
    json_content VARIANT
);

-- STEP 4: Create Stream on Document Stage

CREATE STREAM IF NOT EXISTS my_pdf_stream ON STAGE demo_docs;
ALTER STAGE demo_docs REFRESH;

-- STEP 5: Create Task on Stream for Processing

CREATE OR REPLACE TASK load_new_file_data
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = '1 minutes'
    COMMENT = 'Process new files in the stage and insert data into the pdf_reviews table.'
WHEN SYSTEM$STREAM_HAS_DATA('my_pdf_stream')
AS
INSERT INTO pdf_reviews (
    SELECT
        RELATIVE_PATH AS file_name,
        size AS file_size,
        last_modified,
        file_url AS snowflake_file_url,
        extract_document_data('@demo_docs', RELATIVE_PATH) AS json_content
    FROM my_pdf_stream
    WHERE METADATA$ACTION = 'INSERT'
);

-- Resume the task
ALTER TASK load_new_file_data RESUME;

-- STEP 6: Create Analysis View

CREATE OR REPLACE VIEW pdf_reviews_view AS
SELECT 
    file_name,
    file_size,
    last_modified,
    snowflake_file_url,
    json_content:inspection_date::STRING AS inspection_date,
    json_content:inspection_grade::STRING AS inspection_grade,
    json_content:inspector::STRING AS inspector,
    json_content:list_of_units:unit_name::ARRAY AS list_of_units_name,
    json_content:list_of_units:condition::ARRAY AS list_of_units_condition
FROM pdf_reviews;

-- STEP 7: Create Flattened View (1 Row per Unit)

CREATE OR REPLACE VIEW pdf_reviews_flattened AS
SELECT 
    file_name,
    file_size,
    last_modified,
    snowflake_file_url,
    json_content:inspection_date::STRING AS inspection_date,
    json_content:inspection_grade::STRING AS inspection_grade,
    json_content:inspector::STRING AS inspector,
    f.index AS unit_index,
    f.value::STRING AS unit_name,
    json_content:list_of_units:condition[f.index]::STRING AS unit_condition
FROM pdf_reviews,
LATERAL FLATTEN(input => json_content:list_of_units:unit_name) f;